# Getting started with ipyrowtable

A `RowTable` is an editable table where each row is an element: a room, a cable segment, a
wall layer, a pipe run. **Input columns** are edited by the user; **output columns** are filled
in by a `compute` function you write. Rows can be added and removed, and every change
recomputes immediately.

This notebook builds two small tables from scratch. Run the cells, then edit the tables.

In [ ]:
import numpy as np

from ipyrowtable import (
    ChoiceColumn,
    LiveFigure,
    NodeColumn,
    NumberColumn,
    OutputColumn,
    RowTable,
    TextColumn,
    side_by_side,
)

## 1. The smallest useful table

`compute` receives the inputs and returns a dict of outputs keyed by column. Here
`inputs.column(key)` gives one input column as a NumPy array over the rows.

In [ ]:
def room_areas(inputs):
    return {"area": inputs.column("length") * inputs.column("width")}


rooms = RowTable(
    columns=[
        TextColumn("name", "Room"),
        NumberColumn("length", "Length (m)", default=4.0, min=0),
        NumberColumn("width", "Width (m)", default=3.0, min=0),
        OutputColumn("area", "Area (m²)", fmt="{:.2f}"),
    ],
    compute=room_areas,
    rows=[
        {"name": "Kitchen", "length": 4.2, "width": 3.1},
        {"name": "Bedroom", "length": 3.6, "width": 3.3},
    ],
    item_name="room",
    summary=lambda r: f"Total floor area: <b>{r['area'].sum():.1f} m²</b>",
)
rooms

The latest numbers are always in `.results`. `records()` gives one dict per row, mirroring
the table, which is handy for `pandas.DataFrame(rooms.results.records())`.

In [ ]:
rooms.results.records()

## 2. Values *between* rows: voltage along a cable run

Many problems are elements in series, where the interesting quantity lives at the joints:
temperatures between wall layers, pressures between pipe segments, voltages along a cable.
A `NodeColumn` holds **n + 1** values for **n** rows: one at the start, shown in a leading row
above the first element, then one at the end of each row. Either end can be an editable
boundary condition; here only the source voltage is.

A **parameter** is an input for the whole problem, shown above the table: here, the load
current.

The voltage drop over a segment of a two-wire circuit is `2 · I · R' · L`: the current goes
out and back, so it crosses two conductors.

In [ ]:
# Resistance of copper conductors at 20 °C, ohms per metre of wire
OHM_PER_M = {"14 AWG": 0.008286, "12 AWG": 0.005211, "10 AWG": 0.003277, "8 AWG": 0.002061}


def voltage_drop(inputs):
    ohm_per_m = np.array([OHM_PER_M[g] for g in inputs.column("gauge")])
    drop = 2 * inputs.params["current"] * ohm_per_m * inputs.column("length")
    V_source = inputs.edges["voltage"][0]
    return {"drop": drop, "voltage": V_source - np.concatenate(([0.0], np.cumsum(drop)))}


def drop_summary(results):
    V = results["voltage"]
    percent = 100 * (V[0] - V[-1]) / V[0]
    return f"Voltage at the end of the run: <b>{V[-1]:.1f} V</b> ({percent:.1f}% drop)"


cable = RowTable(
    columns=[
        TextColumn("name", "Segment"),
        ChoiceColumn("gauge", "Wire", options=OHM_PER_M, width="110px"),
        NumberColumn("length", "Length (m)", default=10.0, min=0),
        OutputColumn("drop", "Drop (V)", fmt="{:.2f}"),
        NodeColumn("voltage", "Voltage (V)", inputs="first", first_default=120.0, fmt="{:.1f}"),
    ],
    compute=voltage_drop,
    parameters=[NumberColumn("current", "Load current (A)", default=15.0, min=0, width="90px")],
    rows=[
        {"name": "Panel → junction box", "gauge": "12 AWG", "length": 20},
        {"name": "Junction box → load", "gauge": "14 AWG", "length": 8},
    ],
    leading_label="— source —",
    item_name="segment",
    summary=drop_summary,
)

### Plotting it live

`LiveFigure` redraws a matplotlib figure from the results after every change. You write a
`draw(fig, results)` function; it's hidden while the inputs are invalid.

In [ ]:
def draw_voltage(fig, results):
    ax = fig.add_subplot()
    distance = np.concatenate(([0.0], np.cumsum(results["length"])))
    ax.plot(distance, results["voltage"], marker="o")
    ax.set_xlabel("Distance from source (m)")
    ax.set_ylabel("Voltage (V)")
    ax.set_title("Voltage along the run", loc="left")
    ax.grid(alpha=0.3)


plot = LiveFigure(cable, draw=draw_voltage, size=(5, 3.2))
side_by_side(cable, plot)

## 3. Driving the table from code

Everything you can do with the mouse, you can do from code. Inputs are given in the units
shown in the table.

In [ ]:
cable.add_row({"name": "Load → pump", "gauge": "10 AWG", "length": 15})
cable.param_widget("current").value = 20.0
cable.edge_widget("voltage", "first").value = 240.0

print(f"Voltage at the pump: {cable.results['voltage'][-1]:.1f} V")
print("Rows:", cable.rows)

## Next

* **02_plane_wall_conduction**: a complete application with an SI / Imperial toggle.
* **03_extending**: your own unit systems, a custom column type, and reacting to changes.